# Exploration des sources de données - GlobalTrade Solutions

**Objectif** : Diagnostiquer les fichiers bruts issus des 3 silos (ERP, CRM, Analytics) avant ingestion dans le Lakehouse.

On analyse :
- La structure de chaque fichier (colonnes, types)
- Le volume de données
- Les valeurs manquantes
- Les doublons
- Les clés de jointure entre tables

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("..") / "data" / "raw"

## 1. Silo ERP — Table des ventes (`g_fact_sales.csv`)

In [2]:
sales = pd.read_csv(RAW_DIR / "g_fact_sales.csv")
print(f"Lignes : {len(sales):,}")
print(f"Colonnes : {list(sales.columns)}")
sales.head()

Lignes : 60,398
Colonnes : ['order_number', 'customer_key', 'product_key', 'order_date', 'shipping_date', 'due_date', 'sales', 'quantity', 'price']


,order_number,customer_key,product_key,order_date,shipping_date,due_date,sales,quantity,price
0,SO43697,10769,20,2010-12-29,2011-01-05,2011-01-10,3578,1,3578
1,SO43698,17390,9,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
2,SO43699,14864,9,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
3,SO43700,3502,41,2010-12-29,2011-01-05,2011-01-10,699,1,699
4,SO43701,4,9,2010-12-29,2011-01-05,2011-01-10,3400,1,3400


In [3]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 60398 entries, 0 to 60397
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_number   60398 non-null  str  
 1   customer_key   60398 non-null  int64
 2   product_key    60398 non-null  int64
 3   order_date     60379 non-null  str  
 4   shipping_date  60398 non-null  str  
 5   due_date       60398 non-null  str  
 6   sales          60398 non-null  int64
 7   quantity       60398 non-null  int64
 8   price          60398 non-null  int64
dtypes: int64(5), str(4)
memory usage: 6.3 MB


In [4]:
print("Valeurs manquantes :")
print(sales.isnull().sum())
print(f"\nDoublons (order_number + product_key) : {sales.duplicated(subset=['order_number', 'product_key']).sum()}")

Valeurs manquantes :
order_number      0
customer_key      0
product_key       0
order_date       19
shipping_date     0
due_date          0
sales             0
quantity          0
price             0
dtype: int64

Doublons (order_number + product_key) : 0


In [5]:
sales.describe()

,customer_key,product_key,sales,quantity,price
count,60398.000000,60398.000000,60398.000000,60398.000000,60398.000000
mean,7842.685420,212.284331,486.042336,1.000414,486.058545
std,5432.430404,80.073598,928.452681,0.044011,928.452690
min,1.000000,3.000000,0.000000,1.000000,2.000000
25%,3004.000000,138.000000,8.000000,1.000000,8.000000
50%,7144.000000,246.000000,30.000000,1.000000,30.000000
75%,12430.750000,285.000000,540.000000,1.000000,540.000000
max,18484.000000,295.000000,3578.000000,10.000000,3578.000000


## 2. Silo CRM — Table des clients (`g_dim_customers.csv`)

In [6]:
customers = pd.read_csv(RAW_DIR / "g_dim_customers.csv", encoding="latin-1")
print(f"Lignes : {len(customers):,}")
print(f"Colonnes : {list(customers.columns)}")
customers.head()

Lignes : 18,484
Colonnes : ['customer_key', 'customer_id', 'customer_number', 'first_name', 'last_name', 'country', 'gender', 'marital_status', 'birth_date', 'create_date']


,customer_key,customer_id,customer_number,first_name,last_name,country,gender,marital_status,birth_date,create_date
0,1,11000,AW00011000,Jon,Yang,Australia,Male,Married,1971-10-06,10/6/2025\r
1,2,11001,AW00011001,Eugene,Huang,Australia,Male,Single,1976-05-10,10/6/2025\r
2,3,11002,AW00011002,Ruben,Torres,Australia,Male,Married,1971-02-09,10/6/2025\r
3,4,11003,AW00011003,Christy,Zhu,Australia,Female,Single,1973-08-14,10/6/2025\r
4,5,11004,AW00011004,Elizabeth,Johnson,Australia,Female,Single,1979-08-05,10/6/2025\r


In [7]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 18484 entries, 0 to 18483
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   customer_key     18484 non-null  int64
 1   customer_id      18484 non-null  int64
 2   customer_number  18484 non-null  str  
 3   first_name       18480 non-null  str  
 4   last_name        18481 non-null  str  
 5   country          18147 non-null  str  
 6   gender           18469 non-null  str  
 7   marital_status   18481 non-null  str  
 8   birth_date       18468 non-null  str  
 9   create_date      18484 non-null  str  
dtypes: int64(2), str(8)
memory usage: 2.5 MB


In [8]:
print("Valeurs manquantes :")
print(customers.isnull().sum())
print(f"\nDoublons (customer_key) : {customers.duplicated(subset=['customer_key']).sum()}")
print(f"\nPays représentés : {customers['country'].nunique()}")
customers['country'].value_counts()

Valeurs manquantes :
customer_key         0
customer_id          0
customer_number      0
first_name           4
last_name            3
country            337
gender              15
marital_status       3
birth_date          16
create_date          0
dtype: int64

Doublons (customer_key) : 0

Pays représentés : 6


country
United States     7482
Australia         3591
United Kingdom    1913
France            1810
Germany           1780
Canada            1571
Name: count, dtype: int64

## 3. Silo ERP — Table des produits (`g_dim_products.csv`)

In [9]:
products = pd.read_csv(RAW_DIR / "g_dim_products.csv")
print(f"Lignes : {len(products):,}")
print(f"Colonnes : {list(products.columns)}")
products.head()

Lignes : 295
Colonnes : ['product_key', 'product_id', 'product_number', 'product_name', 'category_id', 'category', 'sub_category', 'maintenance', 'product_ine', 'cost', 'start_date']


,product_key,product_id,product_number,product_name,category_id,category,sub_category,maintenance,product_ine,cost,start_date
0,1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes\r,Road,0,2003-07-01
1,2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes\r,Road,0,2003-07-01
2,3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes\r,Mountain,1898,2011-07-01
3,4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes\r,Mountain,1898,2011-07-01
4,5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes\r,Mountain,1898,2011-07-01


In [10]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   product_key     295 non-null    int64
 1   product_id      295 non-null    int64
 2   product_number  295 non-null    str  
 3   product_name    295 non-null    str  
 4   category_id     295 non-null    str  
 5   category        288 non-null    str  
 6   sub_category    288 non-null    str  
 7   maintenance     288 non-null    str  
 8   product_ine     278 non-null    str  
 9   cost            295 non-null    int64
 10  start_date      295 non-null    str  
dtypes: int64(3), str(8)
memory usage: 47.0 KB


In [11]:
print("Valeurs manquantes :")
print(products.isnull().sum())
print(f"\nDoublons (product_key) : {products.duplicated(subset=['product_key']).sum()}")
print(f"\nCatégories : {products['category'].nunique()}")
products['category'].value_counts()

Valeurs manquantes :
product_key        0
product_id         0
product_number     0
product_name       0
category_id        0
category           7
sub_category       7
maintenance        7
product_ine       17
cost               0
start_date         0
dtype: int64

Doublons (product_key) : 0

Catégories : 4


category
Components     127
Bikes           97
Clothing        35
Accessories     29
Name: count, dtype: int64

## 4. Vérification des clés de jointure entre silos

Le Lakehouse doit relier les 3 silos. Vérifions l'intégrité référentielle.

In [12]:
# Customer keys dans sales mais absents de customers
sales_ckeys = set(sales['customer_key'].unique())
cust_ckeys = set(customers['customer_key'].unique())
orphan_customers = sales_ckeys - cust_ckeys
print(f"customer_key dans sales mais absents de customers : {len(orphan_customers)}")

# Product keys dans sales mais absents de products
sales_pkeys = set(sales['product_key'].unique())
prod_pkeys = set(products['product_key'].unique())
orphan_products = sales_pkeys - prod_pkeys
print(f"product_key dans sales mais absents de products : {len(orphan_products)}")

customer_key dans sales mais absents de customers : 0
product_key dans sales mais absents de products : 0


## 5. Synthèse du diagnostic

| Silo | Table | Lignes | Doublons | Valeurs manquantes | Encodage |
|------|-------|--------|----------|--------------------|----------|
| ERP | g_fact_sales | 60 398 | 0 | 19 (order_date) | UTF-8 |
| CRM | g_dim_customers | 18 484 | 0 | 337 (country), 15 (gender), 16 (birth_date) | Latin-1 |
| ERP | g_dim_products | 295 | 0 | 7 (category, sub_category), 17 (product_ine) | UTF-8 |

**Problèmes identifiés** :
- **Encodage incohérent** : le fichier clients est en Latin-1 alors que les autres sont en UTF-8
- **Caractères parasites** : des `\r` en fin de champs (maintenance, create_date)
- **Valeurs manquantes** : 337 clients sans pays, 19 ventes sans date de commande
- **Typage absent** : les dates sont stockées en chaînes de caractères

**Intégrité référentielle** : les clés de jointure (customer_key, product_key) sont cohérentes entre les 3 tables — 0 clé orpheline.

**Constat** : Les données sont fragmentées en silos sans gouvernance commune. Ces problèmes justifient la mise en place d'une architecture Lakehouse avec couches Bronze (ingestion brute) / Silver (nettoyage, typage, dédoublonnage) / Gold (agrégation KPI).